In [ ]:
import sys
# ensure project root is on sys.path so `import config.config` works
sys.path.append('/home/thibaut/appli/app_oeasc')
sys.path.append('/home/thibaut/appli/app_oeasc/backend')
from pathlib import Path

from app import app

# from utils_flask_sqla.generic import GenericTable
# from flask import request, current_app, jsonify

# from sqlalchemy.exc import SQLAlchemyError
import pandas as pd
import io

from sqlalchemy import func, cast, select, Integer, join, exists
from flask import request, current_app, jsonify
# import models only inside application context
from oeasc.modules.oeasc.commons.models import TEspeces, TSecteurs, TCommunes
from oeasc.modules.oeasc.commons.schema import TEspecesSchema, TSecteursSchema, TCommunesSchema

from oeasc.modules.oeasc.chasse.models import (
    TSaisons,
    TAttributions,
    TTypeBracelets,
    TZoneCynegetiques,
    TZoneIndicatives,
    TAttributionMassifs,
    VPlanChasseRealisationBilan,
    TRealisationsChasse,
    TLieuTirs,
    
)
from oeasc.modules.oeasc.chasse.schema import (
    TAttributionsSchema,
    TRealisationsChasseSchema,
    TSaisonsSchema,
    TTypeBraceletsSchema,
    TLieuTirsSchema,
    TZoneCynegetiquesSchema,
    TZoneIndicativesSchema,
    TAttributionMassifsSchema,
    VPlanChasseRealisationBilanSchema
)

from pypnnomenclature.models import TNomenclatures
from pypnnomenclature.schemas import NomenclatureSchema

with app.app_context():
    config = current_app.config
    DB = config["DB"]


MODE DEVELOPPEMENT ACTIVÉ


In [15]:


path_csv = Path("/home/thibaut/appli/app_oeasc/temp/test1.csv")

if not path_csv.exists():
    raise FileNotFoundError(f"CSV file not found: {path_csv}")

columns_name = [ 
    "id_chasse", "numero", "espece", "age", "sexe", "poids", "pesee", "risque_sanitaire", "type_chasse",
    "ref_battue", "numero_battue", "ref_ug", "ref_detenteur", "nom_detenteur", "ref_equipe", "nom_equipe", 
    "ref_membre", "date", "heure", "commune", "insee_commune", "lieu_dit", "territoire", "departement", 
    "longitude", "latitude", "photos", "commentaires", "nom_tireur", "matin/apres_midi", "tir_plomb",
    "serotheque", "metatarse", "long_pattes", "long_machoire_1", "long_machoire_2", "nb_cors", 
    "nb_tetines", "nb_embryons", "long_cornes_G_isard", "long_cornes_D_isard", "circonf_corne_G_isard", "circonf_corne_D_isard",
    "hauteur_cornes_isard", "ecart_cornes_isard", "age_isard", "long_corne_mouflon", "diam_corne_mouflon", "age_mouflon",
    "parc_nom_tireur", "parc_lieu_dit", "parc_nom_tireur2", "parc_lieu_dit2"
]

df = pd.read_csv(path_csv, sep=";", encoding="utf-8", skiprows=1, names=columns_name, index_col=False)
df.index = df['id_chasse']
df = df.drop(columns=['id_chasse'])
df_trie = df.loc[(df['espece'] != "SANGLIER")]
df_trie = df_trie.set_index('numero')

df_trie

,espece,age,sexe,poids,pesee,risque_sanitaire,type_chasse,ref_battue,numero_battue,ref_ug,...,hauteur_cornes_isard,ecart_cornes_isard,age_isard,long_corne_mouflon,diam_corne_mouflon,age_mouflon,parc_nom_tireur,parc_lieu_dit,parc_nom_tireur2,parc_lieu_dit2
numero,,,,,,,,,,,,,,,,,,,,,
CEFF007043,CERF,Adulte,Femelle,103.00,Plein,Non,Individuel,NaN,NaN,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Agrinier Raphaël,la coustête
CHI006091,CHEVREUIL,Adulte,Male,18.15,Vidé,Non,Collective,4.848048e+18,30.0,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CHI006086,CHEVREUIL,Adulte,Male,16.00,Plein,Non,Individuel,NaN,NaN,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONNIER MICKAEL,TCA D'ALTEFAGE
CEFF006805,CERF,Adulte,Femelle,121.00,Plein,Non,Battue,NaN,NaN,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,burlon bernard,la tourrete
CHI006191,CHEVREUIL,Adulte,Femelle,16.70,Plein,Non,Individuel,NaN,NaN,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Frossard Tom,Volpilloux
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CEFF006706,CERF,Adulte,Femelle,102.00,Plein,Non,Individuel,NaN,NaN,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FOLCHER,Max
CEM006405,CERF,Adulte,Male,230.00,Plein,Non,Individuel,NaN,NaN,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VIVENS,Louis
CEFF006822,CERF,Faon,Femelle,55.00,Plein,Non,Individuel,NaN,NaN,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chaptal Aubin,La Cure


In [16]:
liste_espece = {"CEFF": ["CERF", ["Femelle","Indéterminé", "Male"], ["Adulte", "Subadulte", "Indéterminé", "Faon"]],
                "CEFFD": ["CERF", ["Femelle", "Indéterminé", "Male"], ["Adulte", "Subadulte", "Faon"]],
                "CEM": ["CERF", ["Male"], ["Adulte", "Subadulte", "Indéterminé", "Faon"]],
                "CEMD": ["CERF", ["Male"], ["Jeune"]],

                "MOF": ["MOUFLON", ["Femelle"], ["Adulte", "Jeune"]],
                "MOM": ["MOUFLON", ["Male"], ["Adulte"]],
                "MOM1": ["MOUFLON", ["Male"], ["Adulte", "Subadulte"]],
                "MOI": ["MOUFLON", ["Indéterminé"], ["Jeune"]],
                "MOIJ": ["MOUFLON", ["Femelle"], ["Jeune"]],

                "CHI": ["CHEVREUIL", ["Male", "Indéterminé", "Femelle"], ["Adulte", "Indéterminé", "Jeune"]],
                }

liste_nomenclature_sexe = {"Femelle": 168, "Male": 169, "Indéterminé": 167}
liste_nomenclature_age = {"Adulte": 3, "Subadulte": 6, "Indéterminé": 2, "Jeune": 4, "Faon": 4}
liste_nomenclature_mode_chasse = {"Battue": 575, "Individuel": 574, "Collective": 575, "Affût": 573, "Approche": 574}




In [17]:
id_saison = 43
# on fait une requete à la base de données pour trouver toutes les attributions de la saison id_saison qui n'ont pas de réalisations


with app.app_context():
    # requête anti-existence : garder les attributions sans réalisation
    stmt_attributions = (
        select(TAttributions)
        .where(
            TAttributions.id_saison == id_saison,
            ~exists().where(TRealisationsChasse.id_attribution == TAttributions.id_attribution),
        )
    )
    # retourne des instances ORM de TAttributions
    attributions_non_realisees = DB.session.execute(stmt_attributions).scalars().all()
    attributions_non_realisees_dicts = TAttributionsSchema().dump(attributions_non_realisees, many=True)

df_attributions_non_realisees = pd.DataFrame(attributions_non_realisees_dicts)
df_attributions_non_realisees['numero_bracelet'] = df_attributions_non_realisees['numero_bracelet'].str.replace(" ", "00")
df_attributions_non_realisees.index = df_attributions_non_realisees['numero_bracelet']
df_attributions_non_realisees = df_attributions_non_realisees.drop(columns=['numero_bracelet'])
df_attributions_non_realisees


,id_attribution,saison,zone_cynegetique_affectee,zone_indicative_affectee,type_bracelet,has_realisation,id_type_bracelet,id_saison,id_zone_cynegetique_affectee,id_zone_indicative_affectee,meta_create_date,meta_update_date
numero_bracelet,,,,,,,,,,,,
CEFF006400,15656,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 5, 'secteur': {'id_sec...","{'id_zone_indicative': 16, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,5,16,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CHI005698,15668,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 5, 'secteur': {'id_sec...","{'id_zone_indicative': 16, 'zone_cynegetique':...","{'id_type_bracelet': 1, 'espece': {'id_espece'...",False,1,43,5,16,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CHI005700,15670,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 5, 'secteur': {'id_sec...","{'id_zone_indicative': 16, 'zone_cynegetique':...","{'id_type_bracelet': 1, 'espece': {'id_espece'...",False,1,43,5,16,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEM006243,15679,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 6, 'secteur': {'id_sec...","{'id_zone_indicative': 14, 'zone_cynegetique':...","{'id_type_bracelet': 3, 'espece': {'id_espece'...",False,3,43,6,14,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006608,15682,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 6, 'secteur': {'id_sec...","{'id_zone_indicative': 14, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,6,14,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
...,...,...,...,...,...,...,...,...,...,...,...,...
CEFF006581,17454,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 6, 'secteur': {'id_sec...","{'id_zone_indicative': 18, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,6,18,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006583,17456,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 6, 'secteur': {'id_sec...","{'id_zone_indicative': 18, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,6,18,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006585,17458,"{'id_saison': 43, 'nom_saison': '2023-2024', '...","{'id_zone_cynegetique': 6, 'secteur': {'id_sec...","{'id_zone_indicative': 18, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,6,18,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633


In [18]:
# fusion de df_trie et df_attributions_non_realisees sur l'index (numero_bracelet)
df_fusion = df_trie.join(df_attributions_non_realisees, how='inner')
# df_fusion.to_csv("/home/thibaut/appli/app_oeasc/temp/df_fusion.csv", sep=";", encoding="utf-8", index=True)

df_fusion

,espece,age,sexe,poids,pesee,risque_sanitaire,type_chasse,ref_battue,numero_battue,ref_ug,...,zone_cynegetique_affectee,zone_indicative_affectee,type_bracelet,has_realisation,id_type_bracelet,id_saison,id_zone_cynegetique_affectee,id_zone_indicative_affectee,meta_create_date,meta_update_date
CEFF007043,CERF,Adulte,Femelle,103.0,Plein,Non,Individuel,NaN,NaN,10.0,...,"{'id_zone_cynegetique': 1, 'secteur': {'id_sec...","{'id_zone_indicative': 4, 'zone_cynegetique': ...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,1,4,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006979,CERF,Subadulte,Femelle,85.0,Plein,Non,Individuel,NaN,NaN,10.0,...,"{'id_zone_cynegetique': 1, 'secteur': {'id_sec...","{'id_zone_indicative': 4, 'zone_cynegetique': ...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,1,4,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006976,CERF,Subadulte,Femelle,107.0,Plein,Non,Battue,NaN,NaN,10.0,...,"{'id_zone_cynegetique': 1, 'secteur': {'id_sec...","{'id_zone_indicative': 4, 'zone_cynegetique': ...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,1,4,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006755,CERF,Adulte,Femelle,125.0,Plein,Non,Individuel,NaN,NaN,9.0,...,"{'id_zone_cynegetique': 7, 'secteur': {'id_sec...","{'id_zone_indicative': 25, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,7,25,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CHI006106,CHEVREUIL,Adulte,Male,0.0,Plein,Non,Individuel,NaN,NaN,11.0,...,"{'id_zone_cynegetique': 3, 'secteur': {'id_sec...","{'id_zone_indicative': 17, 'zone_cynegetique':...","{'id_type_bracelet': 1, 'espece': {'id_espece'...",False,1,43,3,17,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CEFF006831,CERF,Adulte,Femelle,127.5,Plein,Non,Individuel,NaN,NaN,11.0,...,"{'id_zone_cynegetique': 1, 'secteur': {'id_sec...","{'id_zone_indicative': 28, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,1,28,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006835,CERF,Faon,Femelle,60.0,Plein,Non,Individuel,NaN,NaN,11.0,...,"{'id_zone_cynegetique': 1, 'secteur': {'id_sec...","{'id_zone_indicative': 15, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,1,15,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006834,CERF,Adulte,Femelle,115.0,Plein,Non,Individuel,NaN,NaN,11.0,...,"{'id_zone_cynegetique': 1, 'secteur': {'id_sec...","{'id_zone_indicative': 15, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,1,15,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633
CEFF006824,CERF,Adulte,Femelle,110.0,Plein,Non,Individuel,NaN,NaN,11.0,...,"{'id_zone_cynegetique': 1, 'secteur': {'id_sec...","{'id_zone_indicative': 28, 'zone_cynegetique':...","{'id_type_bracelet': 2, 'espece': {'id_espece'...",False,2,43,1,28,2023-08-29T14:11:20.473641,2023-08-29T16:20:25.122633


In [ ]:
df_realisations = df_fusion.copy()




# id_realisation
# id_attribution
# id_zone_cynegetique_realisee
# id_zone_indicative_realisee
# id_lieu_tir_synonyme
# date_exacte
# date_enreg
# mortalite_hors_pc
# id_auteur_tir
# id_auteur_constat
# id_nomenclature_sexe
# id_nomenclature_classe_age
# poid_entier
# poid_vide
# poid_c_f_p
# long_dagues_droite
# long_dagues_gauche
# long_mandibules_droite
# long_mandibules_gauche
# cors_nb
# cors_commentaires
# gestation
# id_nomenclature_mode_chasse
# commentaire
# parcelle_onf
# poid_indique
# cors_indetermine
# long_mandibule_indetermine
# id_numerisateur
# meta_create_date
# meta_update_date

In [9]:
with app.app_context():
    stmt = (
        select(TAttributions)
        .where(TAttributions.id_saison == 43)
        .where(~TAttributions.realisations.any())  # Pas de réalisations
    )
    
    attributions_non_realisees = DB.session.execute(stmt).scalars().all()
    
    df_attributions_non_realisees = pd.DataFrame([
        {k: v for k, v in attribution.__dict__.items() if not k.startswith('_')}
        for attribution in attributions_non_realisees
    ])

df_attributions_non_realisees

,id_attribution,numero_bracelet,id_zone_indicative_affectee,meta_update_date,id_type_bracelet,id_saison,id_zone_cynegetique_affectee,meta_create_date,has_realisation
0,15656,CEFF 6400,16,2023-08-29 16:20:25.122633,2,43,5,2023-08-29 14:11:20.473641,False
1,15668,CHI 5698,16,2023-08-29 16:20:25.122633,1,43,5,2023-08-29 14:11:20.473641,False
2,15670,CHI 5700,16,2023-08-29 16:20:25.122633,1,43,5,2023-08-29 14:11:20.473641,False
3,15679,CEM 6243,14,2023-08-29 16:20:25.122633,3,43,6,2023-08-29 14:11:20.473641,False
4,15682,CEFF 6608,14,2023-08-29 16:20:25.122633,2,43,6,2023-08-29 14:11:20.473641,False
...,...,...,...,...,...,...,...,...,...
564,17454,CEFF 6581,18,2023-08-29 16:20:25.122633,2,43,6,2023-08-29 14:11:20.473641,False
565,17456,CEFF 6583,18,2023-08-29 16:20:25.122633,2,43,6,2023-08-29 14:11:20.473641,False
566,17458,CEFF 6585,18,2023-08-29 16:20:25.122633,2,43,6,2023-08-29 14:11:20.473641,False
567,16629,CEFF 7024,4,2025-05-05 17:09:36.096696,2,43,1,2023-08-29 14:11:20.473641,False
